## 1. Configuration

In [1]:
import os, json, time, subprocess
from pathlib import Path
import numpy as np, pandas as pd
import onnxruntime as ort
import torch, torch.nn as nn
import warnings
warnings.filterwarnings("ignore", message="You are using the legacy TorchScript")

MODEL   = "dlinear"
BIT_LEN = "12"          # Deep Prove quantization width (8/10/12/16)
DISTRO  = "Ubuntu"
BENCH   = "/root/deep-prove/target/release/bench"

model_dir = Path("../../models") / MODEL
all_sites = sorted(model_dir.glob("site_*"))
assert all_sites, f"no trained sites under {model_dir}"

work_dir    = Path("_work_dp"); work_dir.mkdir(exist_ok=True)
results_dir = Path("../../results"); results_dir.mkdir(parents=True, exist_ok=True)

def site_id(path): return path.name.replace("site_", "")

def to_wsl_path(p):
    p = Path(p).resolve()
    drive = p.drive.rstrip(":").lower()
    return "/mnt/" + drive + "/" + "/".join(p.parts[1:])

print(len(all_sites), "sites available")

321 sites available


In [2]:
# All three LTSF models are affine, so recover y = W x + b from the ONNX by probing it
# with impulses, then re-export it as one Linear that Deep Prove can prove.
def build_fused_inputs(site_dir):
    meta = json.load(open(site_dir / "meta.json"))
    seq_len, pred_len = meta["seq_len"], meta["pred_len"]
    session = ort.InferenceSession(str(site_dir / "network.onnx"))
    input_name = session.get_inputs()[0].name
    def infer(vec):
        x = vec.reshape(1, seq_len, 1).astype(np.float32)
        return np.array(session.run(None, {input_name: x})[0]).ravel()

    bias = infer(np.zeros(seq_len))
    weight = np.zeros((pred_len, seq_len), np.float32)
    for k in range(seq_len):
        impulse = np.zeros(seq_len); impulse[k] = 1.0
        weight[:, k] = infer(impulse) - bias

    window = np.array(json.load(open(site_dir / "input.json"))["input_data"][0], np.float32)
    recon_err = float(np.abs((weight @ window + bias) - infer(window)).max())

    fused = nn.Linear(seq_len, pred_len)
    with torch.no_grad():
        fused.weight.copy_(torch.from_numpy(weight)); fused.bias.copy_(torch.from_numpy(bias))
    fused.eval()
    onnx_path = work_dir / f"network_dp_{site_id(site_dir)}.onnx"
    torch.onnx.export(fused, torch.zeros(1, seq_len), str(onnx_path),
                      input_names=["input"], output_names=["output"],
                      opset_version=13, dynamo=False)

    # scale the window into [-1, 1], the range Deep Prove expects
    scale = float(np.max(np.abs(window))) or 1.0
    scaled_window = (window / scale).astype(np.float32)
    scaled_out = (weight @ scaled_window + bias).astype(np.float32)
    input_path = work_dir / f"input_dp_{site_id(site_dir)}.json"
    json.dump({"input_data": [scaled_window.tolist()], "output_data": [scaled_out.tolist()],
               "pytorch_output": [scaled_out.tolist()]}, open(input_path, "w"))
    return onnx_path, input_path, meta, recon_err

In [ ]:
# sites = all_sites[:3] run sites

sites = all_sites
print(len(sites), "selected")

321 selected


In [4]:
def prove_site(onnx_path, input_path, csv_path):
    cmd = ["wsl.exe", "-d", DISTRO, "-u", "root", "--",
           "env", f"ZKML_BIT_LEN={BIT_LEN}", BENCH,
           "-o", to_wsl_path(onnx_path), "-i", to_wsl_path(input_path),
           "--bench", to_wsl_path(csv_path), "--num-samples", "1"]
    start = time.perf_counter()
    result = subprocess.run(cmd, capture_output=True, text=True)
    wall_s = time.perf_counter() - start
    if result.returncode != 0:
        print(result.stdout[-1500:]); print(result.stderr[-3000:])
        raise RuntimeError(f"bench failed rc={result.returncode}")
    return wall_s

rows = []
for site_dir in sites:
    onnx_path, input_path, meta, recon_err = build_fused_inputs(site_dir)
    csv_path = work_dir / f"bench_{site_id(site_dir)}.csv"
    if csv_path.exists(): csv_path.unlink()
    wall_s = prove_site(onnx_path, input_path, csv_path)

    bench_df = pd.read_csv(csv_path).set_index("name")
    def metric(name, col="elapsed"):
        return float(bench_df.loc[name, col]) if name in bench_df.index else None
    prove_ms, verify_ms = metric("Proving 0"), metric("Verify 0")

    rows.append({
        "framework": "deepprove", "model": MODEL, "site": meta["site"],
        "params": meta["params"], "bit_len": int(BIT_LEN),
        "prove_s":    round(prove_ms / 1000, 4) if prove_ms is not None else None,
        "verify_s":   round(verify_ms / 1000, 4) if verify_ms is not None else None,
        "inference_s": round((metric("Inference 0") or 0) / 1000, 4),
        "context_s":   round((metric("Context creation") or 0) / 1000, 4),
        "proof_kb":    metric("Proving 0", "proof_size"),
        "peak_mem_mb": round((metric("Proving 0", "peak") or 0) / 1024**2, 1),
        "wall_s":      round(wall_s, 3),
        "affine_recon_err": recon_err,
        "mse_float":   meta["mse_float"],
        "verified":    verify_ms is not None,   # bench aborts on a bad proof, so a Verify row means valid
    })
    print(f"site {site_id(site_dir):>3}: prove={rows[-1]['prove_s']}s verify={rows[-1]['verify_s']}s "
          f"proof={rows[-1]['proof_kb']}KB recon_err={recon_err:.1e}")

site 000: prove=0.936s verify=0.012s proof=40.033203125KB recon_err=2.8e-07


site 001: prove=0.869s verify=0.013s proof=40.033203125KB recon_err=4.8e-07


site 002: prove=0.89s verify=0.012s proof=40.033203125KB recon_err=1.0e-07


site 003: prove=0.788s verify=0.012s proof=40.033203125KB recon_err=3.6e-07


site 004: prove=0.914s verify=0.011s proof=40.033203125KB recon_err=3.6e-07


...

site 316: prove=1.043s verify=0.021s proof=40.033203125KB recon_err=2.4e-07


site 317: prove=0.831s verify=0.018s proof=40.033203125KB recon_err=3.0e-07


site 318: prove=1.032s verify=0.019s proof=40.033203125KB recon_err=6.0e-07


site 319: prove=0.901s verify=0.018s proof=40.033203125KB recon_err=7.2e-07


site  OT: prove=0.862s verify=0.018s proof=40.033203125KB recon_err=4.8e-07


## 5. Save and summarize

`results/deepprove_<model>.csv`, one row per site — same layout as the EZKL CSVs so the analysis
notebook reads both. Remember all three models share the fused circuit, so their Deep Prove rows
are expected to match.

In [5]:
results = pd.DataFrame(rows)
out_path = results_dir / f"deepprove_{MODEL}.csv"
results.to_csv(out_path, index=False)
print("wrote", out_path, "(", len(results), "rows )")
results[["prove_s", "verify_s", "context_s", "proof_kb", "peak_mem_mb"]].describe().round(3)

wrote ..\..\results\deepprove_dlinear.csv ( 321 rows )


,prove_s,verify_s,context_s,proof_kb,peak_mem_mb
count,321.000,321.000,321.000,321.000,321.000
mean,0.899,0.016,0.397,40.033,4171.417
std,0.117,0.003,0.074,0.000,43.112
min,0.721,0.011,0.307,40.033,3470.600
25%,0.835,0.013,0.359,40.033,4174.600
50%,0.870,0.017,0.373,40.033,4174.600
75%,0.923,0.018,0.414,40.033,4174.600
max,1.799,0.046,0.898,40.033,4174.800
